# Milestone 1 — arXiv crawler trên Colab

Notebook này chạy toàn bộ M1: discovery, metadata từng version, tải source TeX/Bib và references từ Semantic Scholar.

## Cách dùng nhanh
1. Chạy cell **1) Cài đặt môi trường** (tự mount Drive, clone/pull code, cài dependency).
2. Chạy cell **2) Nạp cấu hình** và **2b) Scratch SSD**.
3. Chạy cell **3) Xem trạng thái** để biết đang có gì.
4. Chạy cell **4) Chạy crawl**: chọn `RUN_MODE` (`smoke`, `full`, `arxiv_only`, `s2_only`).
5. Khi crawler dừng hẳn, chạy **5) Export** rồi **7) Kiểm tra M1/M2/M3**.

Ghi chú:
- `PROJECT_DIR` mặc định là `/content/drive/MyDrive/arxiv`; phải nằm trên Drive, không dùng ổ tạm.
- Cần `SEMANTIC_SCHOLAR_API_KEY` trong Colab Secrets (bật Notebook access) hoặc biến môi trường.
- Không chạy 2 notebook/runtime cùng lúc trên một dataset.
- Sau khi pull code mới, restart runtime nếu đã import `crawler`.

## 1) Cài đặt môi trường
Chạy một lần sau mỗi lần mở runtime. Cell tự mount Drive, clone hoặc pull code, cài dependency và đưa `src` vào `sys.path`.

In [ ]:
#@title 1) Cài đặt môi trường (chạy 1 lần) { display-mode: "form" }
PROJECT_DIR = "/content/drive/MyDrive/arxiv"  #@param {type:"string"}
REPO_URL = "https://github.com/KienBuiTan-ichica-1/arxiv-kien.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
INSTALL_DEPS = True  #@param {type:"boolean"}

from pathlib import Path
import os, subprocess, sys

try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except ImportError:
    print("Không phải Colab: bỏ qua mount Drive")

root = Path(PROJECT_DIR).expanduser()
if str(root).startswith("/content/") and not str(root).startswith("/content/drive/"):
    raise RuntimeError("PROJECT_DIR phải nằm trên Drive (/content/drive/...), không dùng ổ tạm Colab.")
root.mkdir(parents=True, exist_ok=True)
os.chdir(root)

repo = root / "code"
if not (repo / ".git").exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "remote", "set-url", "origin", REPO_URL], check=True)
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "origin", BRANCH], check=True)

if INSTALL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "src/requirements.txt")], check=True)

sys.path.insert(0, str(repo / "src"))
print("Project root:", root)
print("Code commit:", subprocess.check_output(["git", "-C", str(repo), "log", "-1", "--oneline"], text=True).strip())

## 2) Nạp cấu hình và kiểm tra API key
Đọc `code/config/crawler.yaml`, in range và đường dẫn tuyệt đối của `persistent_root` để chắc chắn dữ liệu ghi lên Drive.

In [ ]:
#@title 2) Nạp cấu hình + kiểm tra API key
from pathlib import Path
import traceback
from crawler.config import load_config
from crawler.pipeline import run_pipeline
from crawler.secrets import load_semantic_scholar_key
from crawler.ids import generate_ids
from crawler.console import log

config = load_config(Path("code/config/crawler.yaml"))
config.execution["allow_anonymous_s2"] = False

def require_s2_key():
    if not load_semantic_scholar_key():
        raise RuntimeError("Thiếu SEMANTIC_SCHOLAR_API_KEY. Thêm trong Colab Secrets, bật Notebook access rồi chạy lại cell này.")

async def run_selected(stages, limit, takeover_stale=False):
    require_s2_key()
    try:
        result = await run_pipeline(config, stages, resume=True, limit=limit, takeover_stale=takeover_stale)
    except BaseException:
        traceback.print_exc()
        raise
    print(result)
    return result

input_ids = generate_ids(**config.range)
print("Range:", input_ids[0], "→", input_ids[-1], "| Tổng:", len(input_ids))
print("Persistent root (absolute):", Path(config.paths["persistent_root"]).resolve())
print("Scratch root:", Path(config.paths["scratch_root"]).resolve())
require_s2_key()
print("S2 key: đã tìm thấy (chưa kiểm chứng quyền truy cập API)")

In [ ]:
#@title 2b) Dùng SSD tạm cho scratch (khuyến nghị) { display-mode: "form" }
USE_LOCAL_SCRATCH = True  #@param {type:"boolean"}
if USE_LOCAL_SCRATCH:
    import tempfile
    scratch_dir = Path("scratch")
    try:
        if scratch_dir.is_symlink() and not scratch_dir.exists():
            scratch_dir.unlink()
        if not scratch_dir.exists():
            scratch_dir.symlink_to(tempfile.mkdtemp(prefix="arxiv-crawler-"), target_is_directory=True)
        config.paths["scratch_root"] = "scratch"
        config.validate()
        print("Scratch ->", Path(config.paths["scratch_root"]).resolve())
    except OSError as exc:
        print("Không tạo được symlink scratch trên Drive:", exc)
        print("Giữ scratch_root mặc định:", config.paths["scratch_root"])
else:
    print("Giữ scratch_root trong YAML:", config.paths["scratch_root"])

## 3) Xem trạng thái hiện tại
Chỉ đọc dữ liệu đã lưu, không gọi API, không reset job/cooldown. Dùng lại hàm `print_crawl_progress` trong cell chạy crawl.

In [ ]:
#@title 3) Xem trạng thái hiện tại
from crawler.progress import ProgressStore
from datetime import datetime, timezone
import json, time

def _elapsed_text(seconds):
    h, rem = divmod(int(seconds), 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

def read_counts():
    try:
        return ProgressStore(config.paths["persistent_root"]).status_counts()
    except Exception as exc:
        print(f"[progress] chưa đọc được: {type(exc).__name__}: {exc}")
        return None

def print_crawl_progress(total_papers, started=None):
    counts = read_counts()
    if counts is None:
        return
    d = counts["discovery"]
    terminal = sum(d.get(s, 0) for s in ("succeeded", "no_tex", "not_found", "failed"))
    pct = (terminal / total_papers * 100) if total_papers else 0
    clock = time.strftime("%H:%M:%S")
    elapsed = f" | đã {_elapsed_text(time.time() - started)}" if started else ""
    detail = " ".join([
        "ok=" + str(d.get("succeeded", 0)),
        "not_found=" + str(d.get("not_found", 0)),
        "failed=" + str(d.get("failed", 0)),
        "running=" + str(d.get("running", 0)),
        "pending=" + str(d.get("pending", 0)),
    ])
    print(f"[{clock}]{elapsed} | discovery {terminal}/{total_papers} ({pct:.1f}%) | {detail}", flush=True)
    for stage in ("metadata", "source", "references"):
        c = counts[stage]
        stage_detail = " ".join(f"{k}={v}" for k, v in sorted(c.items()))
        print(f"    {stage:<10} {stage_detail}", flush=True)

root = Path(config.paths["persistent_root"])
progress = root / "progress"
if not (progress / "jobs.csv").exists() and not (progress / "jobs.bak").exists():
    print("Chưa có progress tại:", progress.resolve())
else:
    print_crawl_progress(len(input_ids))
    store = ProgressStore(root)
    failures = [r for r in store.rows.values() if r["status"] == "failed"]
    print("Số job failed:", len(failures))
    for row in failures[:20]:
        print({k: row[k] for k in ("paper", "version", "stage", "offset", "attempts", "error")})
    if len(failures) > 20:
        print("Chỉ hiển thị 20 lỗi đầu.")

services = progress / "services.json"
if services.exists():
    for service, state in json.loads(services.read_text()).items():
        deadline = state.get("deadline", 0)
        print(service, "cooldown đến:", datetime.fromtimestamp(deadline, timezone.utc).isoformat(),
              "| còn chờ (giây):", round(max(0, deadline - time.time())),
              "| backoff:", state.get("backoff", 0))

## 4) Chạy crawl
Chọn chế độ trong form. Cell in tiến độ mỗi `PROGRESS_EVERY_SECONDS` giây. Bấm **Stop/Interrupt** sẽ huỷ pipeline, chờ lưu progress rồi mới thoát.

- `smoke`: 10 ID đầu, arXiv + S2.
- `full`: toàn range, arXiv + S2.
- `arxiv_only`: toàn range, chỉ arXiv.
- `s2_only`: toàn range, chỉ S2 (dùng khi arXiv đã discovery).

`LIMIT = 0` nghĩa là dùng limit mặc định của mode (smoke=10, còn lại toàn range). Đặt `LIMIT > 0` để giới hạn thêm.

In [ ]:
#@title 4) Chạy crawl { display-mode: "form" }
RUN_MODE = "full"  #@param ["smoke", "full", "arxiv_only", "s2_only"]
LIMIT = 0  #@param {type:"integer"}
PROGRESS_EVERY_SECONDS = 60  #@param {type:"integer"}
TAKEOVER_STALE = False  #@param {type:"boolean"}

import asyncio, time

mode = RUN_MODE.strip().lower()
if mode == "smoke":
    stages, limit = {"arxiv", "s2"}, 10
elif mode == "full":
    stages, limit = {"arxiv", "s2"}, None
elif mode == "arxiv_only":
    stages, limit = {"arxiv"}, None
elif mode == "s2_only":
    stages, limit = {"s2"}, None
else:
    raise ValueError("RUN_MODE không hợp lệ: " + RUN_MODE)
if LIMIT and LIMIT > 0:
    limit = LIMIT

total_papers = len(input_ids[:limit]) if limit else len(input_ids)
started = time.time()
print(f"Chạy mode={mode} | stages={sorted(stages)} | limit={limit} | takeover={TAKEOVER_STALE}", flush=True)

task = asyncio.create_task(run_selected(stages, limit, takeover_stale=TAKEOVER_STALE))
try:
    while not task.done():
        await asyncio.to_thread(print_crawl_progress, total_papers, started)
        await asyncio.wait({task}, timeout=PROGRESS_EVERY_SECONDS)
    summary = await task
except BaseException:
    print("Đang huỷ pipeline và chờ lưu progress...", flush=True)
    task.cancel()
    try:
        await task
    except BaseException:
        pass
    raise

print_crawl_progress(total_papers, started)
print("Hoàn tất. Summary:")
print(summary)

## 5) Export snapshot
Chỉ chạy khi crawler đã dừng. Export tạo `runtime/exports/<run_id>/` gồm `papers.parquet`, `references.parquet`, metrics CSV và manifest.

In [ ]:
#@title 5) Export snapshot { display-mode: "form" }
EXPORT_SNAPSHOT = False  #@param {type:"boolean"}
RUN_ID = ""  #@param {type:"string"}
import json, uuid
if EXPORT_SNAPSHOT:
    from crawler.export import export_snapshot
    run_id = RUN_ID.strip() or ("export-" + uuid.uuid4().hex)
    manifest = export_snapshot(config, run_id)
    print(json.dumps(manifest, indent=2, ensure_ascii=False))
else:
    print("Bỏ qua: EXPORT_SNAPSHOT=False")

## 6) Audit checksum
Kiểm tra lại checksum của kết quả đã publish. Không gọi API, không reset trạng thái crawl. Chạy khi crawler đã dừng.

In [ ]:
#@title 6) Audit checksum { display-mode: "form" }
RUN_AUDIT = False  #@param {type:"boolean"}
FULL_AUDIT = False  #@param {type:"boolean"}
TAKEOVER_STALE = False  #@param {type:"boolean"}
if RUN_AUDIT:
    from crawler.verification import audit
    print(audit(config, ids=set(generate_ids(**config.range)),
                full=FULL_AUDIT, takeover_stale=TAKEOVER_STALE))
else:
    print("Bỏ qua: RUN_AUDIT=False")

## 7) Kiểm tra dữ liệu với yêu cầu M1/M2/M3
Đối chiếu nhanh dữ liệu đã crawl/export với đặc tả:

- **M1:** đủ discovery/metadata/source/references; `papers.parquet` và `references.parquet` đúng cột; chỉ giữ `.tex`/`.bib` trong `data/`.
- **M2:** cần `elements`, `hierarchy`, `bibtex` — repo này chưa triển khai.
- **M3:** cần `predictions` và pipeline matching MRR@5 — repo này chưa triển khai.

Cell này chỉ đọc file local, không gọi API.

In [ ]:
#@title 7) Kiểm tra M1/M2/M3
import json
from pathlib import Path
from crawler.progress import ProgressStore

PAPERS_COLUMNS = ["arxiv_id", "title", "authors", "submission_date", "revised_dates", "venue"]
REFERENCES_COLUMNS = ["arxiv_id", "ref_arxiv_id", "ref_title", "ref_authors", "ref_submission_date", "ref_s2_id"]
try:
    import pyarrow.parquet as pq
except ImportError:
    pq = None

root = Path(config.paths["persistent_root"]).resolve()
print("M1 — Dữ liệu crawl")
store = ProgressStore(root)
counts = store.status_counts()
total = len(generate_ids(**config.range))
for stage in ("discovery", "metadata", "source", "references"):
    print(f"  {stage:<11}", counts.get(stage, {}))

exports = root / "exports"
export_dirs = sorted([p for p in exports.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime) if exports.exists() else []
if not export_dirs:
    print("  Chưa có export. Chạy cell 5) Export trước để kiểm tra schema Parquet.")
else:
    latest = export_dirs[-1]
    print("  Export mới nhất:", latest)
    manifest_path = latest / "manifest.json"
    if manifest_path.exists():
        m = json.loads(manifest_path.read_text())
        print("  manifest: papers=", m.get("papers"), "references=", m.get("references"),
              "missing_metadata=", m.get("missing_metadata"), "schema_version=", m.get("schema_version"))
    if pq is None:
        print("  pyarrow chưa cài: bỏ qua kiểm tra cột Parquet.")
    else:
        for name, expected in [("papers.parquet", PAPERS_COLUMNS), ("references.parquet", REFERENCES_COLUMNS)]:
            path = latest / name
            if not path.exists():
                print(f"  THIẾU {name}")
                continue
            table = pq.read_table(path)
            cols = table.column_names
            missing = [c for c in expected if c not in cols]
            extra = [c for c in cols if c not in expected]
            print(f"  {name}: rows={table.num_rows} missing={missing} extra={extra}")

data = root / "data"
bad = []
if data.exists():
    for p in data.rglob("*"):
        if p.is_file() and p.suffix.lower() not in (".tex", ".bib", ".json", ".xml"):
            bad.append(p)
    print("  File ngoài .tex/.bib/.json/.xml trong data/:", len(bad))
    for p in bad[:10]:
        print("   ", p.relative_to(root))

print("")
print("M2 — Parsing: CHƯA có trong repo này (chưa có elements/hierarchy/bibtex).")
print("M3 — Matching: CHƯA có trong repo này (chưa có predictions, chưa train MRR@5).")
print("Kết luận: repo hiện đạt phần M1 (crawl + export papers/references); M2/M3 cần triển khai thêm.")

## 8) Chẩn đoán source
Chỉ đọc payload lỗi còn giữ trong scratch, không gọi API, không reset failed.

In [ ]:
#@title 8) Chẩn đoán source
from crawler.diagnostics import diagnose_sources
import json
print(json.dumps(diagnose_sources(config), indent=2, ensure_ascii=False))

## 9) Đo tổng dung lượng (tùy chọn)
Quét toàn bộ cây persistent/scratch. Có thể rất lâu trên Drive; không cần cho resume.

In [ ]:
#@title 9) Đo tổng dung lượng { display-mode: "form" }
MEASURE_TOTAL_DISK = False  #@param {type:"boolean"}
TAKEOVER_STALE = False  #@param {type:"boolean"}
if MEASURE_TOTAL_DISK:
    from crawler.disk_usage import DiskUsage
    from crawler.locking import DatasetLock
    import uuid
    with DatasetLock(config.paths["persistent_root"], "disk-" + uuid.uuid4().hex, TAKEOVER_STALE):
        print("Đang quét toàn bộ dung lượng; Drive có thể rất chậm...", flush=True)
        total_bytes = DiskUsage(config.paths.values()).total
        print("Tổng logical bytes:", total_bytes, flush=True)
else:
    print("Bỏ qua: MEASURE_TOTAL_DISK=False")

## Ghi chú vận hành

### Dữ liệu nằm ở đâu
- Kết quả crawl: `runtime/data/<arxiv_id>/` (chỉ `.tex`/`.bib` cùng JSON metadata).
- Progress: `runtime/progress/` (`jobs.csv`, `jobs.journal.jsonl`, `papers.csv`, `versions.csv`).
- Export: `runtime/exports/<run_id>/`.
- `scratch/` chỉ là vùng tạm, bị xoá sau publish; không tìm paper ở đó.

### Lỗi “Previous active marker exists”
Nghĩa là lần chạy trước chưa thoát sạch, còn `runtime/progress/active_run.json`. Chỉ bật `TAKEOVER_STALE = True` sau khi đã restart/terminate runtime cũ và chắc chắn không còn process nào ghi dataset; chạy một lần rồi đổi lại `False`.

### Không nên
- Không xoá `jobs.journal.jsonl`, cooldown hay hạ code cũ trên dataset đã có journal.
- Không chạy hai notebook/runtime cùng lúc trên cùng `persistent_root`.
- Không đổi `persistent_root` khi đang resume cùng dataset.